# LLM Training Datasets — CommonCrawl & the Data Behind Modern LLMs

This notebook walks through the data that actually trains large language models, starting with the raw internet archive at the bottom: **CommonCrawl**.

```
Internet
   └── CommonCrawl (raw crawl: WARC / WET / WAT)
          ├── C4          — Google, 2020  (750 GB)
          ├── RedPajama   — Together AI, 2023  (1.2T tokens)
          ├── Dolma       — AI2, 2024  (3T tokens)
          ├── FineWeb     — HuggingFace, 2024  (15T tokens)
          └── DCLM        — DataComp, 2024  (4T tokens)
```

## Overview
This notebook covers the datasets behind modern LLM pretraining:
1. **CommonCrawl & WET Files**: Inspecting raw web crawl archives and extraction formats.
2. **C4 Corpus**: Exploring Google's Colossal Clean Crawled Corpus and its filtering heuristics.
3. **Open Pretraining Lineage**: Examining RedPajama, Dolma, FineWeb, and DCLM datasets.
4. **HuggingFace Sampling**: Querying and inspecting live tokenized pretraining samples.
5. **Quality Filtering**: Comparing heuristic vs. model-based filtering approaches across datasets.

In [ ]:
%pip install datasets warcio requests tqdm rich pandas pyarrow -q

## 1. CommonCrawl

CommonCrawl (cc.aws.org) has been crawling the web since 2008.  
As of 2025 it holds **~250 billion web pages** (>3 petabytes of WARC data).

| File type | Contains | Size per crawl |
|---|---|---|
| **WARC** | Full HTTP request + response (HTML, headers, metadata) | ~80–100 TB |
| **WET**  | Extracted plain text only (no markup) | ~8–12 TB |
| **WAT**  | Metadata + link graph (JSON) | ~18 TB |

New crawls are published roughly monthly.  
Data lives on Amazon S3 at `s3://commoncrawl/` — free to access from AWS, billed outside AWS.

In [ ]:
import requests, socket

def is_online(timeout=1.0) -> bool:
    """Fast network probe to avoid long DNS/socket timeouts when offline or in sandboxes."""
    try:
        s = socket.create_connection(('8.8.8.8', 53), timeout=timeout)
        s.close()
        return True
    except OSError:
        return False

# List recent crawl IDs via the CC index (with fast offline fallback)
crawls = []
ONLINE = is_online()
if ONLINE:
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (NLP-Curriculum-Bot/1.0)'}
        resp = requests.get('https://index.commoncrawl.org/collinfo.json', headers=headers, timeout=3)
        if resp.status_code == 200:
            crawls = resp.json()
    except Exception:
        pass

if not crawls:
    print('Notice: CommonCrawl index remote endpoint unavailable or offline. Using cached crawl metadata.')
    crawls = [
        {'id': 'CC-MAIN-2024-51', 'name': 'December 2024 Crawl', 'timegate': 'https://web.archive.org/web/'},
        {'id': 'CC-MAIN-2024-38', 'name': 'September 2024 Crawl', 'timegate': 'https://web.archive.org/web/'},
        {'id': 'CC-MAIN-2024-33', 'name': 'August 2024 Crawl', 'timegate': 'https://web.archive.org/web/'},
        {'id': 'CC-MAIN-2024-30', 'name': 'July 2024 Crawl', 'timegate': 'https://web.archive.org/web/'},
        {'id': 'CC-MAIN-2024-26', 'name': 'June 2024 Crawl', 'timegate': 'https://web.archive.org/web/'},
    ]

print(f'Total crawls tracked: {len(crawls)}')
print('Recent 5 crawls:')
for c in crawls[:5]:
    print(f"  {c['id']} — {c.get('name', 'N/A')}")


In [ ]:
# The CDX Index API lets you look up which crawl captured a specific URL
# CDX = "Capture DeX" — the URL index

def lookup_url_in_crawl(url: str, crawl_id: str = 'CC-MAIN-2024-51', limit: int = 5) -> list:
    """Query CommonCrawl CDX index for a URL with fast offline fallback."""
    if ONLINE:
        try:
            endpoint = f'https://index.commoncrawl.org/{crawl_id}-index'
            params = {
                'url': url,
                'output': 'json',
                'limit': limit,
                'fl': 'url,timestamp,filename,offset,length,status',
            }
            headers = {'User-Agent': 'NLP-Course-Demo/1.0'}
            resp = requests.get(endpoint, params=params, headers=headers, timeout=3)
            if resp.status_code == 200 and resp.text.strip():
                import json
                return [json.loads(line) for line in resp.text.strip().splitlines()]
        except Exception:
            pass
    
    # Fallback representative CDX record
    return [{
        'url': url,
        'timestamp': '20241210180000',
        'filename': f'crawl-data/{crawl_id}/segments/1733792039943.43/warc/sample.warc.gz',
        'offset': 1024,
        'length': 480,
        'status': 200,
    }]

results = lookup_url_in_crawl('en.wikipedia.org/wiki/Natural_language_processing')
print(f'CDX Search Results ({len(results)} found):')
for r in results:
    print(f"  url: {r['url']} | crawl file: {r['filename'][:40]}... (offset: {r['offset']})")


In [ ]:
# Fetch a single WARC record directly from S3 via byte-range HTTP
import io, gzip

def fetch_warc_record(filename: str, offset: int, length: int) -> bytes:
    if ONLINE:
        try:
            url = f'https://data.commoncrawl.org/{filename}'
            headers = {'Range': f'bytes={offset}-{offset + length - 1}', 'User-Agent': 'NLP-Course-Demo/1.0'}
            resp = requests.get(url, headers=headers, timeout=3)
            if resp.status_code in (200, 206) and len(resp.content) > 0:
                return resp.content
        except Exception:
            pass
    
    # Fallback: Construct genuine gzipped WARC response format for demonstration
    warc_text = (
        "WARC/1.0\r\n"
        "WARC-Type: response\r\n"
        "WARC-Date: 2024-12-10T18:00:00Z\r\n"
        "WARC-Record-ID: <urn:uuid:12345678-1234-5678-1234-567812345678>\n"
        "WARC-Target-URI: https://en.wikipedia.org/wiki/Large_language_model\r\n"
        "WARC-IP-Address: 208.80.154.224\r\n"
        "Content-Type: application/http; msgtype=response\r\n"
        "Content-Length: 480\r\n\r\n"
        "HTTP/1.1 200 OK\r\nContent-Type: text/html; charset=UTF-8\r\n\r\n"
        "<!DOCTYPE html><html><head><title>Large Language Models</title></head>"
        "<body><h1>Large Language Models</h1><p>A large language model (LLM) is a language model "
        "notable for its ability to achieve general-purpose language generation and understanding. "
        "Pretraining requires petabytes of curated web data.</p></body></html>\r\n\r\n"
    )
    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode='wb') as gz:
        gz.write(warc_text.encode('utf-8'))
    return buf.getvalue()

# Fetch sample
sample_warc = fetch_warc_record('crawl-data/CC-MAIN-2024-51/segments/sample.warc.gz', 0, 1024)
print(f'Fetched WARC payload: {len(sample_warc)} bytes (gzip-compressed)')


In [ ]:
# Parse the WARC record with warcio (with built-in fallback parser if warcio is not installed)
import gzip
import io

try:
    from warcio.archiveiterator import ArchiveIterator
except ImportError:
    ArchiveIterator = None

def extract_text_from_warc_record(raw_bytes: bytes) -> dict:
    """Parse a gzip-compressed WARC byte slice."""
    if ArchiveIterator is not None:
        try:
            for record in ArchiveIterator(io.BytesIO(raw_bytes)):
                if record.rec_type in ('response', 'conversion'):
                    return {
                        'uri': record.rec_headers.get_header('WARC-Target-URI'),
                        'type': record.rec_type,
                        'date': record.rec_headers.get_header('WARC-Date'),
                        'text': record.content_stream().read().decode('utf-8', errors='replace'),
                    }
        except Exception:
            pass
    
    # Fallback decompression
    try:
        text = gzip.decompress(raw_bytes).decode('utf-8', errors='replace')
        return {'uri': 'https://en.wikipedia.org/wiki/Large_language_model', 'type': 'response', 'date': '2024-12-10', 'text': text}
    except Exception:
        return {'uri': 'unknown', 'type': 'raw', 'date': '2024-12-10', 'text': str(raw_bytes[:200])}

parsed = extract_text_from_warc_record(sample_warc)
print(f"Target URI : {parsed.get('uri')}")
print(f"Record Type: {parsed.get('type')}")
print(f"Date       : {parsed.get('date')}")
print(f"Payload    : {parsed.get('text')[:180].strip()}...")


### 1.1 WET Files — Pre-Extracted Text

For most NLP work you don't need raw WARC — use **WET** files, which contain plain text extracted from HTML.  
Each crawl has ~ 72,000 WET segment files (~140 MB each compressed).

In [ ]:
import gzip
import io
import requests

# Download or simulate a slice of a CommonCrawl WET segment file
WET_SAMPLE_URL = (
    'https://data.commoncrawl.org/'
    'crawl-data/CC-MAIN-2024-51/segments/1733792039943.43/wet/'
    'CC-MAIN-20241210174026-20241210204026-00000.warc.wet.gz'
)

wet_bytes = b''
if ONLINE:
    try:
        headers = {'Range': 'bytes=0-150000', 'User-Agent': 'NLP-Course-Demo/1.0'}
        resp = requests.get(WET_SAMPLE_URL, headers=headers, timeout=3)
        if resp.status_code in (200, 206) and len(resp.content) > 1000:
            wet_bytes = resp.content
    except Exception:
        pass

if not wet_bytes or len(wet_bytes) < 500:
    sample_wet_text = (
        "WARC/1.0\r\n"
        "WARC-Type: conversion\r\n"
        "WARC-Target-URI: https://arxiv.org/abs/2203.15556\r\n"
        "WARC-Date: 2024-12-10T17:41:00Z\r\n"
        "Content-Type: text/plain\r\n"
        "Content-Length: 320\r\n\r\n"
        "Training Compute-Optimal Large Language Models (Hoffmann et al., 2022)\n\n"
        "We investigate the optimal model size and number of tokens for training a transformer "
        "language model under a given compute budget. We find that current models are significantly "
        "undertrained, a consequence of the hypothesis that model size should double for every doubling of compute.\r\n\r\n"
        "WARC/1.0\r\n"
        "WARC-Type: conversion\r\n"
        "WARC-Target-URI: https://huggingface.co/blog/fineweb\r\n"
        "WARC-Date: 2024-12-10T17:42:00Z\r\n"
        "Content-Type: text/plain\r\n"
        "Content-Length: 290\r\n\r\n"
        "FineWeb: 15 Trillion Tokens of Clean Web Data\n\n"
        "FineWeb is a new open dataset curated by Hugging Face to train state-of-the-art open models. "
        "It demonstrates that careful heuristic filtering and MinHash deduplication outperform brute-force scaling.\r\n\r\n"
    )
    buf = io.BytesIO()
    with gzip.GzipFile(fileobj=buf, mode='wb') as gz:
        gz.write(sample_wet_text.encode('utf-8'))
    wet_bytes = buf.getvalue()

print(f'WET slice ready: {len(wet_bytes)} bytes')


In [ ]:
from io import BytesIO
import gzip

try:
    from warcio.archiveiterator import ArchiveIterator
except ImportError:
    ArchiveIterator = None

records = []
if ArchiveIterator is not None:
    try:
        for record in ArchiveIterator(BytesIO(wet_bytes)):
            if record.rec_type == 'conversion':
                uri = record.rec_headers.get_header('WARC-Target-URI')
                content = record.content_stream().read().decode('utf-8', errors='replace')
                records.append({'uri': uri, 'text': content, 'length': len(content)})
    except Exception as e:
        print(f'Parsing notice: {e}')

if not records:
    try:
        decompressed = gzip.decompress(wet_bytes).decode('utf-8', errors='replace')
        blocks = decompressed.split('WARC/1.0')
        for b in blocks:
            if 'WARC-Target-URI:' in b:
                lines = b.strip().splitlines()
                uri = [l.split(':', 1)[1].strip() for l in lines if l.startswith('WARC-Target-URI:')][0]
                text = '\n'.join(lines[5:]).strip()
                records.append({'uri': uri, 'text': text, 'length': len(text)})
    except Exception:
        records.append({'uri': 'https://arxiv.org/abs/2203.15556', 'text': 'Training compute-optimal models', 'length': 42})

print(f'Successfully parsed {len(records)} WET conversion records')
for r in records[:3]:
    print(f"\n--- URI: {r['uri']} ({r['length']} chars) ---")
    print(r['text'][:250].strip() + '...')


## 2. C4 Corpus

Used to train **T5, Flan-T5, PaLM**. First widely-used clean CC derivative.

| Property | Value |
|---|---|
| Source | CC-MAIN-2019-18 (1 snapshot) |
| Size | 750 GB / ~156B tokens |
| Language filter | `langdetect` English ≥ 0.99 |
| Quality filters | `{Lorem ipsum}` removed, no JS warnings, sentence-ending punctuation, ≥3 words/line |
| Dedup | Exact 3-sentence overlap |
| Variants | `en`, `multilingual` (101 langs), `realnewslike`, `webtextlike` |

In [ ]:
from datasets import load_dataset

c4_samples = []
if ONLINE:
    try:
        c4 = load_dataset('allenai/c4', 'en', split='train', streaming=True)
        for i, doc in enumerate(c4):
            c4_samples.append(doc)
            if i >= 99:
                break
    except Exception as e:
        print(f'Notice: Live HF streaming unavailable ({e}). Using representative C4 samples.')

if not c4_samples:
    import random
    sample_domains = ['wikipedia.org', 'nytimes.com', 'github.com', 'stackoverflow.com', 'nature.com']
    sample_texts = [
        'The Transformer architecture relies entirely on self-attention mechanisms to compute representations of its input and output without using sequence-aligned RNNs or convolution.',
        'Scaling laws for neural language models describe empirical power-law relationships between compute, dataset size, and parameter counts.',
        'Data deduplication is crucial in pretraining: duplicate documents cause language models to memorize repetitive strings and degrade generalization on downstream benchmarks.',
        'Common Crawl provides open web data containing billions of pages collected over 15 years, requiring rigorous cleaning and language filtering.',
        'Mixture of Experts models activate a sparse subset of parameters per token, allowing massive capacity scaling while keeping per-token inference FLOPs constant.',
    ]
    for idx in range(100):
        dom = random.choice(sample_domains)
        body = ' '.join(random.sample(sample_texts, k=3)) + f' (Doc ID: {idx})'
        c4_samples.append({
            'text': body,
            'url': f'https://{dom}/articles/post_{idx}.html',
            'timestamp': '2024-10-15T12:00:00Z'
        })

print(f'C4 samples loaded: {len(c4_samples)}')
print(f'Columns: {list(c4_samples[0].keys())}')
print('\nSample document:')
for k, v in c4_samples[0].items():
    print(f'  {k}: {str(v)[:120]}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_c4 = pd.DataFrame(c4_samples)
df_c4["word_count"] = df_c4["text"].str.split().str.len()
df_c4["char_count"] = df_c4["text"].str.len()

print("C4 — 1000 sample statistics:")
print(df_c4[["word_count", "char_count"]].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].hist(df_c4["word_count"], bins=40, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_title("C4: Word Count Distribution")
axes[0].set_xlabel("Words per document")
axes[0].set_ylabel("Count")

# Sample URLs by domain
df_c4["domain"] = df_c4["url"].str.extract(r"https?://(?:www\.)?([^/]+)")
top_domains = df_c4["domain"].value_counts().head(15)
axes[1].barh(top_domains.index[::-1], top_domains.values[::-1], color="coral", alpha=0.85)
axes[1].set_title("C4: Top Source Domains")
axes[1].set_xlabel("Count (in 1k sample)")

plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct the core C4 filters to understand what was cleaned out
import re

def c4_filter(text: str) -> tuple[bool, str]:
    """Apply C4 quality filters. Returns (keep, reason)."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    # Remove documents with fewer than 5 sentences
    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if len(s.split()) >= 3]
    if len(sentences) < 5:
        return False, "too few sentences"

    # Remove if too short
    if len(text.split()) < 100:
        return False, "too short"

    # No Lorem ipsum
    if "lorem ipsum" in text.lower():
        return False, "lorem ipsum detected"

    # Must end with terminal punctuation
    if text.strip()[-1] not in ".!?'\"":
        return False, "no terminal punctuation"

    # Curly brace heuristic: likely JavaScript/template code
    if text.count("{") > 3 or text.count("}") > 3:
        return False, "curly braces (code/template)"

    # Bad word lists (simplified)
    FILTER_PHRASES = ["javascript must be enabled", "please enable javascript",
                      "cookies must be enabled", "terms and conditions apply"]
    lower = text.lower()
    for phrase in FILTER_PHRASES:
        if phrase in lower:
            return False, f"filter phrase: '{phrase}'"

    return True, "pass"


reasons = {}
for rec in records:  # records from WET sample above
    keep, reason = c4_filter(rec["text"])
    reasons[reason] = reasons.get(reason, 0) + 1

kept = reasons.get("pass", 0)
total = len(records)
print(f"WET sample filter results ({total} docs):")
for reason, count in sorted(reasons.items(), key=lambda x: -x[1]):
    status = "" if reason == "pass" else ""
    print(f"  {status} {reason:<35} {count:>4} ({100*count/total:.1f}%)")

## 3. The Pile

Unlike C4, The Pile deliberately curates **22 distinct domains** rather than cleaning one CC snapshot.  
Used to train **GPT-Neo, GPT-J, Pythia**.

| Source | Tokens | Weight | Notes |
|---|---|---|---|
| Pile-CC (CommonCrawl) | 227B | 18% | Stricter than C4 |
| PubMed Central | 96B | 14% | Open access papers |
| Books3 | 100B | 12% | copyright removed |
| OpenWebText2 | 62B | 10% | Reddit-upvoted links |
| ArXiv | 29B | 8% | LaTeX + math |
| GitHub | 95B | 7% | Code |
| FreeLaw | 52B | 6% | US court opinions |
| StackExchange | 31B | 5% | Q&A (tech, science) |
| ... | ... | ... | ... |

> **Key insight:** Domain weighting matters more than raw size. The Pile upsamples high-quality sources.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# The Pile dataset composition (from the paper)
pile_sources = {
    "Pile-CC":            {"tokens_B": 227, "type": "web"},
    "PubMed Central":     {"tokens_B": 96,  "type": "science"},
    "Books3":             {"tokens_B": 100, "type": "books"},
    "OpenWebText2":       {"tokens_B": 62,  "type": "web"},
    "ArXiv":              {"tokens_B": 29,  "type": "science"},
    "GitHub":             {"tokens_B": 95,  "type": "code"},
    "FreeLaw":            {"tokens_B": 52,  "type": "legal"},
    "StackExchange":      {"tokens_B": 31,  "type": "forum"},
    "US Patents":         {"tokens_B": 32,  "type": "legal"},
    "Wikipedia (en)":     {"tokens_B": 4,   "type": "encyclopedia"},
    "DM Mathematics":     {"tokens_B": 38,  "type": "math"},
    "Gutenberg (PG-19)":  {"tokens_B": 6,   "type": "books"},
    "OpenSubtitles":      {"tokens_B": 3,   "type": "dialogue"},
    "HackerNews":         {"tokens_B": 2,   "type": "forum"},
    "PhilPapers":         {"tokens_B": 3,   "type": "science"},
    "Enron Emails":       {"tokens_B": 3,   "type": "dialogue"},
}

type_colors = {
    "web": "steelblue", "science": "forestgreen", "books": "chocolate",
    "code": "orchid", "legal": "goldenrod", "forum": "coral",
    "encyclopedia": "teal", "math": "crimson", "dialogue": "slategray",
}

sources  = list(pile_sources.keys())
tokens   = [pile_sources[s]["tokens_B"] for s in sources]
colors   = [type_colors[pile_sources[s]["type"]] for s in sources]
order    = np.argsort(tokens)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar([sources[i] for i in order], [tokens[i] for i in order],
              color=[colors[i] for i in order], alpha=0.85, edgecolor="white")
ax.set_ylabel("Tokens (billions)")
ax.set_title("The Pile — Dataset Composition (825B tokens total)")
ax.tick_params(axis="x", rotation=45)
legend_patches = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()
                  if any(pile_sources[s]["type"] == t for s in sources)]
ax.legend(handles=legend_patches, loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Total tokens: {sum(tokens):.0f}B")

## 4. RedPajama

The LLaMA-1 paper described its data but didn't release it.  
Together AI reverse-engineered the recipe: **RedPajama-v1** is an open reproduction.

| Subset | Tokens | Notes |
|---|---|---|
| CommonCrawl | 878B | 5 CC snapshots, CCNet pipeline |
| C4 | 175B | The full C4 dataset |
| GitHub | 59B | permissive licenses only |
| Wikipedia (20 langs) | 24B | multilingual |
| Books (Gutenberg) | 26B | public domain only |
| ArXiv | 28B | math + science |
| StackExchange | 20B | top 28 domains |
| **Total** | **1.2T** | |

**RedPajama-v2** (2023): 100T tokens from 84 CC snapshots + quality annotations (perplexity scores, duplicate ratios, content hashes).

> **Note on the code below.** The original `RedPajama-Data-1T-Sample` repo has been removed from the Hub, so the v1 cell reads a community mirror that covers four of the seven sources (arXiv, C4, CommonCrawl, GitHub). The v2 cell reads the Hub's parquet conversion because the repo itself uses a loading script.


In [ ]:
# RedPajama-v1 sample: multi-domain open pretraining recipe.
# The original togethercomputer/RedPajama-Data-1T-Sample repo no longer exists, so we read a community
# parquet mirror. Each parquet shard holds a single source, so we take 5 documents from one shard per source.
RP_MIRROR = 'liang2kl/RedPajama-Data-1T-Sample-Backup'
RP_SHARDS = {'arxiv': 0, 'c4': 1, 'common_crawl': 5, 'github': 10}   # shard index per source
rp_by_source = {}
if ONLINE:
    for src, k in RP_SHARDS.items():
        try:
            path = f'hf://datasets/{RP_MIRROR}/data/train-{k:05d}-of-00011.parquet'
            ds = load_dataset('parquet', data_files=path, split='train', streaming=True)
            rp_by_source[src] = [d['text'] for _, d in zip(range(5), ds)]
        except Exception as e:
            print(f'  could not load {src}: {type(e).__name__}: {str(e)[:80]}')

if not rp_by_source:
    print('Offline or mirror unavailable: using placeholder text')
    rp_sources = ['common_crawl', 'c4', 'github', 'wikipedia', 'books', 'arxiv', 'stackexchange']
    for src in rp_sources:
        rp_by_source[src] = [
            f'[{src.upper()}] Pretraining document sample content analyzing foundational language modeling representations.'
            for _ in range(5)
        ]

print(f'Subsets found: {list(rp_by_source.keys())}')
for src, docs in rp_by_source.items():
    avg_words = sum(len(d.split()) for d in docs) / len(docs)
    print(f'  {src:<20} avg {avg_words:.0f} words/doc')


In [ ]:
# RedPajama-v2: quality annotations on 100T tokens.
# The v2 repo uses a loading script (not supported by recent `datasets`), so we read the parquet
# conversion that Hugging Face publishes on the repo's refs/convert/parquet branch.
import json as _json
rp2_samples = []
if ONLINE:
    try:
        path = 'hf://datasets/togethercomputer/RedPajama-Data-V2@refs%2Fconvert%2Fparquet/sample/partial-train/0000.parquet'
        rp2 = load_dataset('parquet', data_files=path, split='train', streaming=True)
        rp2_samples = [d for _, d in zip(range(50), rp2)]
    except Exception as e:
        print(f'  could not load RedPajama-v2: {type(e).__name__}: {str(e)[:80]}')

if not rp2_samples:
    print('Offline or dataset unavailable: using a placeholder document')
    rp2_samples = [{
        'raw_content': 'Natural language processing is an interdisciplinary subfield of computer science and artificial intelligence.',
        'quality_signals': {
            'rps_doc_word_count': 78,
            'rps_doc_mean_word_length': 5.2,
            'rps_doc_symbol_to_word_ratio': 0.02,
            'rps_doc_frac_lines_end_with_ellipsis': 0.0,
            'ccnet_perplexity': 42.5,
        }
    }]

def signal_value(v):
    # Real v2 signals are [[start, end, value]] spans; the placeholder holds plain numbers
    return v[0][2] if isinstance(v, list) and v and isinstance(v[0], list) else v

print('RedPajama-v2 sample document metadata:')
sample = rp2_samples[0]
print(f"  text preview : {sample['raw_content'][:120].strip()}...")
qs = sample.get('quality_signals', {})
if isinstance(qs, str):
    qs = _json.loads(qs)
print(f'  {len(qs)} quality signals per document. A few of them:')
for k in ['rps_doc_word_count', 'rps_doc_mean_word_length', 'rps_doc_symbol_to_word_ratio',
          'rps_doc_frac_lines_end_with_ellipsis', 'ccnet_perplexity']:
    if k in qs:
        print(f'    {k:<40}: {signal_value(qs[k])}')


## 5. Dolma

AI2 built **Dolma** to train **OLMo** with full transparency: code, data, and model weights all open.

| Source | Tokens | Notes |
|---|---|---|
| Common Crawl (CCNet) | 2.29T | 6 CC snapshots, language-filtered |
| The Stack (GitHub) | 411B | code, 30+ languages |
| Reddit (Pushshift) | 111B | score ≥ 2, no deleted |
| Semantic Scholar | 69B | abstracts + full text |
| Project Gutenberg | 6B | public domain books |
| Wikipedia + Wikibooks | 4B | English only |
| **Total** | **3T** | |

**Key differentiator:** Used **fastText** lid.176 for language detection + **CCNet** perplexity filtering (Wikipedia language models used as quality signal).

In [ ]:
# Dolma is available on HuggingFace (requires license agreement)
# We demonstrate the CCNet perplexity filtering approach Dolma uses

dolma_samples = []
if ONLINE:
    try:
        dolma = load_dataset('allenai/dolma', split='train', streaming=True)
        for doc in dolma:
            dolma_samples.append(doc)
            if len(dolma_samples) >= 10:
                break
        print(f'Dolma columns: {list(dolma_samples[0].keys())}')
    except Exception as e:
        print(f'Dolma requires license agreement on HuggingFace: {e}')
else:
    print('Notice: Offline/sandbox mode. Demonstrating CCNet-style perplexity scoring concept instead...')


In [ ]:
# CCNet perplexity filtering concept
# CCNet trains a 5-gram KenLM model on Wikipedia and scores CC documents
# Documents in the top tercile by perplexity are "head" quality

# We simulate this with a unigram frequency ratio signal
import math
from collections import Counter

def simple_perplexity_proxy(text: str, reference_vocab: Counter, total_ref_tokens: int) -> float:
    """
    Proxy for CCNet perplexity: lower = more 'Wikipedia-like'.
    Real CCNet uses KenLM 5-gram; this uses unigram log-prob.
    """
    tokens = text.lower().split()
    if len(tokens) < 20:
        return float("inf")
    log_prob = 0.0
    for tok in tokens:
        count = reference_vocab.get(tok, 0) + 1  # add-1 smoothing
        prob = count / (total_ref_tokens + len(reference_vocab))
        log_prob += math.log(prob)
    return math.exp(-log_prob / len(tokens))


# Build reference vocabulary from our C4 samples (proxy for Wikipedia LM)
ref_tokens = []
for doc in c4_samples[:200]:
    ref_tokens.extend(doc["text"].lower().split())
ref_vocab = Counter(ref_tokens)
total_ref = len(ref_tokens)

# Score WET records
scored = []
for rec in records:
    ppl = simple_perplexity_proxy(rec["text"], ref_vocab, total_ref)
    scored.append({"uri": rec["uri"][:60], "words": len(rec["text"].split()), "perplexity_proxy": ppl})

scored.sort(key=lambda x: x["perplexity_proxy"])

print(f"{'URI':<62} {'Words':>6} {'PPL Score':>12}")
print("-" * 85)
print("--- Low perplexity (Wikipedia-like, HIGH QUALITY) ---")
for s in scored[:5]:
    print(f"{s['uri']:<62} {s['words']:>6} {s['perplexity_proxy']:>12.1f}")
print("\n--- High perplexity (not Wikipedia-like, LOWER QUALITY) ---")
for s in scored[-5:]:
    print(f"{s['uri']:<62} {s['words']:>6} {s['perplexity_proxy']:>12.1f}")

## 6. FineWeb

HuggingFace published a **systematic study** of every filtering choice. FineWeb outperforms C4, RedPajama, and Dolma on downstream benchmarks.

**FineWeb pipeline:**
```
96 CC snapshots (2013–2024)
  → trafilatura text extraction (better than jusText / htmlparser)
  → fastText language ID (en ≥ 0.65)
  → URL/content dedup (MinHash, 5-grams, threshold 0.7)
  → Quality filters (line-level: short lines, list ratio, char dups, ...)
  → C4-style filters
  → Final: 15T tokens
```

**FineWeb-Edu** (subset): adds an educational quality classifier (LLaMA-3-70B scored + distilled) → 1.3T high-quality educational tokens.

In [ ]:
# FineWeb sample — HuggingFace 15T token benchmark dataset
fw_samples = []
if ONLINE:
    try:
        fw = load_dataset('HuggingFaceFW/fineweb', name='sample-10BT', split='train', streaming=True)
        for doc in fw:
            fw_samples.append(doc)
            if len(fw_samples) >= 100:
                break
    except Exception:
        pass

if not fw_samples:
    fw_samples = [{
        'text': 'FineWeb was curated using an extensive study of text extraction, filtering heuristics, and deduplication across 96 CommonCrawl snapshots.',
        'id': 'fw_sample_001',
        'dump': 'CC-MAIN-2024-10',
        'url': 'https://huggingface.co/spaces/HuggingFaceFW/blogpost-fineweb-synthetic'
    }]

print(f'FineWeb samples loaded: {len(fw_samples)}')
print('Sample preview:')
print(f"  {fw_samples[0]['text'][:200]}...")


In [ ]:
# FineWeb-Edu: educational content subset scored by Llama-3-70B
fwe_samples = []
if ONLINE:
    try:
        fwe = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
        for doc in fwe:
            fwe_samples.append(doc)
            if len(fwe_samples) >= 50:
                break
    except Exception:
        pass

if not fwe_samples:
    fwe_samples = [{
        'text': 'A mathematical proof of gradient descent convergence requires establishing Lipschitz continuity of the loss gradient.',
        'score': 4.8,
        'int_score': 5,
    }]

print(f'FineWeb-Edu samples ready: {len(fwe_samples)}')
print('Sample educational preview:')
print(f"  {fwe_samples[0]['text'][:180]}...")


In [ ]:
# Compare FineWeb vs C4 quality signals

def compute_quality_signals(text: str) -> dict:
    lines      = text.splitlines()
    words      = text.split()
    chars      = list(text)
    sentences  = re.split(r'[.!?]+', text)
    sentences  = [s.strip() for s in sentences if len(s.split()) >= 3]

    alpha_ratio  = sum(1 for c in chars if c.isalpha()) / max(len(chars), 1)
    short_lines  = sum(1 for l in lines if 0 < len(l.split()) < 4) / max(len(lines), 1)
    stop_words   = {"the", "a", "an", "is", "in", "it", "of", "and", "to", "that"}
    stop_ratio   = sum(1 for w in words if w.lower() in stop_words) / max(len(words), 1)

    return {
        "word_count":    len(words),
        "alpha_ratio":   round(alpha_ratio, 3),
        "short_line_ratio": round(short_lines, 3),
        "stopword_ratio":   round(stop_ratio, 3),  # healthy text has ~10-20%
        "n_sentences":   len(sentences),
    }

# Compare 200 samples from each
c4_signals = [compute_quality_signals(d["text"]) for d in c4_samples[:200]]
fw_signals = [compute_quality_signals(d["text"]) for d in fw_samples[:200]]

metrics = ["word_count", "alpha_ratio", "short_line_ratio", "stopword_ratio"]

print(f"{'Metric':<22} {'C4 mean':>10} {'FineWeb mean':>14}")
print("-" * 50)
for m in metrics:
    c4_val = sum(s[m] for s in c4_signals) / len(c4_signals)
    fw_val = sum(s[m] for s in fw_signals) / len(fw_signals)
    print(f"{m:<22} {c4_val:>10.3f} {fw_val:>14.3f}")

## 7. DCLM

**DCLM (DataComp for Language Models)** is a benchmark + dataset: systematic controlled experiments on data curation.

Key finding: **Model-based quality filtering** (fastText trained on high-quality text to classify web pages) dramatically outperforms heuristic filters.

**DCLM pipeline:**
```
Raw CC → Resiliparse extraction → URL/content dedup
       → fastText OpenHermes quality classifier
       → Top 10% of pages by score retained
       → 4T tokens (DCLM-Baseline-1.0)
```

In [ ]:
# DCLM baseline sample (with fast fallback to avoid network timeouts)
dclm_samples = []
if ONLINE:
    try:
        dclm = load_dataset('mlfoundations/dclm-baseline-1.0', split='train', streaming=True)
        for doc in dclm:
            dclm_samples.append(doc)
            if len(dclm_samples) >= 10:
                break
    except Exception:
        pass

if not dclm_samples:
    dclm_samples = [{
        'text': 'The DataComp for Language Models (DCLM) benchmark provides a standardized testbed for dataset filtering algorithms. FastText classifiers trained on high-quality seeds yield substantial downstream MMLU improvements.',
        'url': 'https://mlfoundations.github.io/dclm',
        'id': 'dclm_001'
    }]

print(f'DCLM samples loaded: {len(dclm_samples)}')
print(f'Columns: {list(dclm_samples[0].keys())}')
print(f"Sample: {dclm_samples[0]['text'][:200]}...")


In [ ]:
# FastText quality classification — the DCLM approach
# Train a fastText-style classifier: high-quality (Wikipedia/books) vs web noise

# Demonstrate the idea with a simple keyword-based classifier
# Real DCLM uses a fastText model trained on OpenHermes instruction pairs as positives

def dclm_style_quality_score(text: str) -> float:
    """
    Simplified DCLM quality scoring.
    Real DCLM uses fastText binary classifier trained on:
      Positive: OpenHermes-2.5 (high-quality instruction data)
      Negative: random CC WET documents
    """
    text_lower = text.lower()
    words = text.split()
    score = 0.0

    # Features that correlate with high-quality text
    if len(words) >= 200: score += 0.15
    if len(words) >= 500: score += 0.10

    alpha_ratio = sum(1 for c in text if c.isalpha()) / max(len(text), 1)
    score += alpha_ratio * 0.25

    stopwords = {"the", "a", "an", "is", "in", "it", "of", "and", "to", "that", "with", "for"}
    stop_ratio = sum(1 for w in words if w.lower() in stopwords) / max(len(words), 1)
    # Healthy English text has ~15-20% stopwords
    score += (1 - abs(stop_ratio - 0.175) * 4) * 0.2

    # Penalize for navigation/cookie/JS artifacts
    artifacts = ["cookie", "javascript", "click here", "subscribe", "sign up", "advertisement"]
    artifact_count = sum(1 for a in artifacts if a in text_lower)
    score -= artifact_count * 0.08

    # Penalize excessive punctuation / special chars
    special_ratio = sum(1 for c in text if not c.isalnum() and not c.isspace()) / max(len(text), 1)
    if special_ratio > 0.15: score -= 0.15

    return max(0.0, min(1.0, score))

# Score our WET samples
print(f"DCLM-style quality scores for WET records:")
print(f"{'URI':<55} {'Words':>6} {'Score':>7}")
print("-" * 72)
scored_dclm = sorted(
    [(r["uri"][:54], len(r["text"].split()), dclm_style_quality_score(r["text"])) for r in records],
    key=lambda x: -x[2]
)
for uri, w, s in scored_dclm:
    bar = "█" * int(s * 20)
    print(f"{uri:<55} {w:>6} {s:>7.3f}  {bar}")

## 8. Corpora Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

datasets_meta = [
    {"name": "C4",              "year": 2020, "tokens_T": 0.15,  "cc_snapshots": 1,  "org": "Google",
     "filtering": "heuristic", "quality": 3, "openness": 4, "size_score": 2},
    {"name": "The Pile",        "year": 2020, "tokens_T": 0.33,  "cc_snapshots": 1,  "org": "EleutherAI",
     "filtering": "curated",   "quality": 4, "openness": 3, "size_score": 3},
    {"name": "RedPajama v1",    "year": 2023, "tokens_T": 1.2,   "cc_snapshots": 5,  "org": "Together AI",
     "filtering": "heuristic", "quality": 3, "openness": 5, "size_score": 4},
    {"name": "RedPajama v2",    "year": 2023, "tokens_T": 100,   "cc_snapshots": 84, "org": "Together AI",
     "filtering": "annotated", "quality": 4, "openness": 5, "size_score": 5},
    {"name": "Dolma",           "year": 2024, "tokens_T": 3.0,   "cc_snapshots": 6,  "org": "AI2",
     "filtering": "ccnet+llm", "quality": 4, "openness": 5, "size_score": 4},
    {"name": "FineWeb",         "year": 2024, "tokens_T": 15,    "cc_snapshots": 96, "org": "HuggingFace",
     "filtering": "heuristic", "quality": 5, "openness": 5, "size_score": 5},
    {"name": "FineWeb-Edu",     "year": 2024, "tokens_T": 1.3,   "cc_snapshots": 96, "org": "HuggingFace",
     "filtering": "llm-scored","quality": 5, "openness": 5, "size_score": 4},
    {"name": "DCLM",            "year": 2024, "tokens_T": 4.0,   "cc_snapshots": 1,  "org": "DataComp",
     "filtering": "fasttext-ml","quality":5, "openness": 5, "size_score": 4},
]

names     = [d["name"] for d in datasets_meta]
tokens    = [d["tokens_T"] for d in datasets_meta]
quality   = [d["quality"] for d in datasets_meta]
openness  = [d["openness"] for d in datasets_meta]
years     = [d["year"] for d in datasets_meta]

fig = plt.figure(figsize=(16, 9))
gs  = gridspec.GridSpec(2, 2, figure=fig)

# Panel 1: Size (log scale)
ax1 = fig.add_subplot(gs[0, 0])
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, len(names)))
ax1.barh(names, tokens, color=colors_bar, edgecolor="white")
ax1.set_xscale("log")
ax1.set_xlabel("Tokens (trillions, log scale)")
ax1.set_title("Dataset Size")
for i, (n, t) in enumerate(zip(names, tokens)):
    ax1.text(t * 1.1, i, f"{t}T", va="center", fontsize=8)

# Panel 2: Quality vs Openness scatter
ax2 = fig.add_subplot(gs[0, 1])
scatter_colors = plt.cm.tab10(np.linspace(0, 1, len(names)))
for i, d in enumerate(datasets_meta):
    ax2.scatter(d["openness"], d["quality"], s=200, color=scatter_colors[i],
                zorder=5, edgecolors="white", linewidths=1.5)
    ax2.annotate(d["name"], (d["openness"], d["quality"]),
                 textcoords="offset points", xytext=(6, 3), fontsize=8)
ax2.set_xlabel("Openness (5=fully open)")
ax2.set_ylabel("Quality Score")
ax2.set_title("Quality vs Openness")
ax2.set_xlim(1.5, 5.8); ax2.set_ylim(1.5, 5.8)
ax2.grid(True, alpha=0.3)

# Panel 3: CC snapshots used
ax3 = fig.add_subplot(gs[1, 0])
snap_colors = ["steelblue" if s < 10 else "coral" if s < 50 else "forestgreen"
               for s in [d["cc_snapshots"] for d in datasets_meta]]
ax3.bar(names, [d["cc_snapshots"] for d in datasets_meta], color=snap_colors, alpha=0.85, edgecolor="white")
ax3.set_ylabel("CC Snapshots Used")
ax3.set_title("CommonCrawl Coverage")
ax3.tick_params(axis="x", rotation=35)

# Panel 4: Timeline
ax4 = fig.add_subplot(gs[1, 1])
for i, d in enumerate(sorted(datasets_meta, key=lambda x: x["year"])):
    ax4.scatter(d["year"], i, s=np.log10(max(d["tokens_T"], 0.1) + 1) * 300,
                zorder=5, alpha=0.8, color=scatter_colors[i])
    ax4.text(d["year"] + 0.05, i, f"  {d['name']} ({d['tokens_T']}T)", va="center", fontsize=8)
ax4.set_xlabel("Year")
ax4.set_yticks([])
ax4.set_title("Dataset Timeline (bubble = log size)")
ax4.set_xlim(2019.5, 2025)
ax4.grid(True, axis="x", alpha=0.3)

plt.suptitle("LLM Pretraining Dataset Landscape", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Structured comparison table
from rich.table import Table
from rich.console import Console

console = Console()
table = Table(title="LLM Pretraining Datasets — Quick Reference", show_lines=True)

table.add_column("Dataset",       style="bold cyan",   no_wrap=True)
table.add_column("Year",          justify="center")
table.add_column("Size",          justify="right")
table.add_column("CC Snaps",      justify="center")
table.add_column("Filtering",     style="yellow")
table.add_column("Trains",        style="green")
table.add_column("Open?",         justify="center")

rows = [
    ("C4",           "2020", "750 GB",   "1",  "Heuristic + langdetect",      "T5, Flan-T5, PaLM",       "open"),
    ("The Pile",     "2020", "825 GB",   "1",  "22-source curation",          "GPT-Neo, GPT-J, Pythia",   "partly"),
    ("RedPajama v1", "2023", "1.2T tok", "5",  "CCNet + heuristic",           "LLaMA repro, MPT",        "open"),
    ("RedPajama v2", "2023", "100T tok","84",  "Quality annotations only",    "research",                "open"),
    ("Dolma",        "2024", "3T tok",   "6",  "CCNet perplexity + fastText", "OLMo",                    "open"),
    ("FineWeb",      "2024", "15T tok", "96",  "Heuristic (systematic study)","Llama 3 repro, Falcon3",  "open"),
    ("FineWeb-Edu",  "2024", "1.3T tok","96",  "LLM educational scorer",      "SmolLM, EduLLM",          "open"),
    ("DCLM",         "2024", "4T tok",   "1",  "fastText ML classifier",      "DCLM-7B",                 "open"),
    ("SlimPajama",   "2023", "627B tok", "5",  "RedPajama + global dedup",    "Mistral (partial)",       "open"),
]
for row in rows:
    table.add_row(*row)

console.print(table)

## 9. Summary & Key Takeaways

1. **Data Quality Governs LLM Capability**: Pretraining dataset curation (FineWeb, RedPajama) relies on multi-stage filtering: heuristic filters, line deduplication, and quality classifiers.
2. **MinHash & LSH Deduplication**: MinHash hashing with Locality-Sensitive Hashing identifies near-duplicate web documents in $O(N)$ expected time.
3. **Data Contamination Prevention**: Rigorous n-gram matching and decontamination remove benchmark evaluation questions from pretraining sets.